# 09 Train RL Overlay SAC v1

This notebook calls `train_rl_overlay_sac_v1.py` to train, validate, and test the SAC overlay for monthly portfolio allocation. The notebook is intentionally thin: all production logic lives in the Python script.

In [1]:
from pathlib import Path
import shlex
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
SCRIPT = PROJECT_ROOT / "train_rl_overlay_sac_v1.py"

print(f"Python executable: {sys.executable}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Script: {SCRIPT}")

Python executable: /opt/anaconda3/envs/portfolio_allocation_rl/bin/python
Project root: /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl
Script: /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/train_rl_overlay_sac_v1.py


## Configuration

Edit the parameters below before launching a full training run. For a quick smoke test, reduce `TOTAL_TIMESTEPS` and narrow the date ranges.

In [2]:
OUTDIR = "data/rl_overlay_sac"

TRAIN_START = "2006-01"
TRAIN_END = "2016-12"
VAL_START = "2017-01"
VAL_END = "2019-12"
TEST_START = "2020-01"
TEST_END = "2025-12"

TOTAL_TIMESTEPS = 200_000
EVAL_FREQUENCY = 5_000
COST_BPS = 10.0
MAX_WEIGHT = None
SOLVER = "CLARABEL"
SEED = 42

USE_VECNORMALIZE = True
WARM_START_CVXPY = True

FF_FILE = None

## Preflight Checks

In [ ]:
required_paths = [
    SCRIPT,
    PROJECT_ROOT / "data/prediction/fm_oos_predictions.parquet",
    PROJECT_ROOT / "data/risk/risk_cov_npz",
    PROJECT_ROOT / "data/risk/risk_monthly_metadata.csv",
    PROJECT_ROOT / "data/panel/monthly_stock_panel_basic_full.parquet",
]

missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required paths:\n" + "\n".join(str(path) for path in missing)
    )

print("All required paths found.")

All required paths found.


## Build Command

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT),
    "--project-root",
    str(PROJECT_ROOT),
    "--outdir",
    OUTDIR,
    "--train-start",
    TRAIN_START,
    "--train-end",
    TRAIN_END,
    "--val-start",
    VAL_START,
    "--val-end",
    VAL_END,
    "--test-start",
    TEST_START,
    "--test-end",
    TEST_END,
    "--total-timesteps",
    str(TOTAL_TIMESTEPS),
    "--eval-frequency",
    str(EVAL_FREQUENCY),
    "--cost-bps",
    str(COST_BPS),
    "--solver",
    SOLVER,
    "--seed",
    str(SEED),
    "--use-vecnormalize",
    str(USE_VECNORMALIZE).lower(),
    "--warm-start-cvxpy",
    str(WARM_START_CVXPY).lower(),
]

if MAX_WEIGHT is not None:
    cmd.extend(["--max-weight", str(MAX_WEIGHT)])

if FF_FILE is not None:
    cmd.extend(["--ff-file", str(FF_FILE)])

print(" ".join(shlex.quote(part) for part in cmd))

/opt/anaconda3/envs/portfolio_allocation_rl/bin/python /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/train_rl_overlay_sac_v1.py --project-root /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl --outdir data/rl_overlay_sac --train-start 2006-01 --train-end 2016-12 --val-start 2017-01 --val-end 2019-12 --test-start 2020-01 --test-end 2025-12 --total-timesteps 200000 --eval-frequency 5000 --cost-bps 10.0 --solver CLARABEL --seed 42 --use-vecnormalize true --warm-start-cvxpy true


## Run Training Pipeline

In [6]:
result = subprocess.run(cmd, cwd=PROJECT_ROOT, check=False)
if result.returncode != 0:
    raise RuntimeError(f"Training script failed with return code {result.returncode}.")

2026-04-28 22:42:37 INFO train_rl_overlay_sac_v1 - Using pyarrow 20.0.0 for parquet IO.
2026-04-28 22:42:37 INFO train_rl_overlay_sac_v1 - Loading prediction file from /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/prediction/fm_oos_predictions.parquet.
2026-04-28 22:42:37 INFO train_rl_overlay_sac_v1 - Kept 100,092 of 100,092 prediction rows after cleaning.
2026-04-28 22:42:37 INFO train_rl_overlay_sac_v1 - Loading return panel from /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/panel/monthly_stock_panel_basic_full.parquet.
2026-04-28 22:42:37 INFO train_rl_overlay_sac_v1 - Kept 139,179 of 139,179 return rows after cleaning.
2026-04-28 22:42:37 INFO train_rl_overlay_sac_v1 - Loading risk metadata from /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/risk/risk_monthly_metadata.csv.
2026-04-28 22:42:37 INFO train_rl_overlay_sac_v1 - Discovered 228 covariance files in /Users/lava/Documents/Research

Using cpu device


2026-04-28 22:42:41 INFO train_rl_overlay_sac_v1 - Starting SAC training for 200,000 timesteps.
2026-04-28 22:42:46 WARNING train_rl_overlay_sac_v1 - 2009-12-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-28 22:42:48 WARNING train_rl_overlay_sac_v1 - 2011-12-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-28 22:42:49 WARNING train_rl_overlay_sac_v1 - 2012-03-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-28 22:42:49 WARNING train_rl_overlay_sac_v1 - 2012-04-30 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-28 22:42:50 WARNING train_rl_overlay_sac_v1 

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 132      |
|    ep_rew_mean     | 0.807    |
|    success_rate    | 1        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 9        |
|    time_elapsed    | 57       |
|    total_timesteps | 528      |
| train/             |          |
|    actor_loss      | -4.48    |
|    critic_loss     | 0.0577   |
|    ent_coef        | 0.88     |
|    ent_coef_loss   | -0.429   |
|    learning_rate   | 0.0003   |
|    n_updates       | 427      |
---------------------------------


2026-04-28 22:43:44 WARNING train_rl_overlay_sac_v1 - 2009-12-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-28 22:43:47 WARNING train_rl_overlay_sac_v1 - 2011-12-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-28 22:43:47 WARNING train_rl_overlay_sac_v1 - 2012-03-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-28 22:43:47 WARNING train_rl_overlay_sac_v1 - 2012-04-30 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-28 22:43:48 WARNING train_rl_overlay_sac_v1 - 2012-11-30 next-month returns are partially missing for optimizer holdings; renormalizing over

...

2026-04-29 07:14:29 WARNING train_rl_overlay_sac_v1 - 2009-12-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-29 07:14:32 WARNING train_rl_overlay_sac_v1 - 2011-12-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-29 07:14:32 WARNING train_rl_overlay_sac_v1 - 2012-03-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-29 07:14:32 WARNING train_rl_overlay_sac_v1 - 2012-04-30 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-29 07:14:33 WARNING train_rl_overlay_sac_v1 - 2012-11-30 next-month returns are partially missing for optimizer holdings; renormalizing over

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 132      |
|    ep_rew_mean     | 5.17     |
|    success_rate    | 1        |
| time/              |          |
|    episodes        | 1508     |
|    fps             | 6        |
|    time_elapsed    | 30761    |
|    total_timesteps | 199056   |
| train/             |          |
|    actor_loss      | -2.03    |
|    critic_loss     | 3.36e-05 |
|    ent_coef        | 0.00267  |
|    ent_coef_loss   | 0.727    |
|    learning_rate   | 0.0003   |
|    n_updates       | 198955   |
---------------------------------


2026-04-29 07:15:28 WARNING train_rl_overlay_sac_v1 - 2009-12-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-29 07:15:30 WARNING train_rl_overlay_sac_v1 - 2011-12-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-29 07:15:31 WARNING train_rl_overlay_sac_v1 - 2012-03-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-29 07:15:31 WARNING train_rl_overlay_sac_v1 - 2012-04-30 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-29 07:15:31 WARNING train_rl_overlay_sac_v1 - 2012-11-30 next-month returns are partially missing for optimizer holdings; renormalizing over

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 132      |
|    ep_rew_mean     | 5.17     |
|    success_rate    | 1        |
| time/              |          |
|    episodes        | 1512     |
|    fps             | 6        |
|    time_elapsed    | 30817    |
|    total_timesteps | 199584   |
| train/             |          |
|    actor_loss      | -1.96    |
|    critic_loss     | 7.3e-05  |
|    ent_coef        | 0.00265  |
|    ent_coef_loss   | 1.98     |
|    learning_rate   | 0.0003   |
|    n_updates       | 199483   |
---------------------------------


2026-04-29 07:16:23 WARNING train_rl_overlay_sac_v1 - 2009-12-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-29 07:16:26 WARNING train_rl_overlay_sac_v1 - 2011-12-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-29 07:16:26 WARNING train_rl_overlay_sac_v1 - 2012-03-31 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-29 07:16:26 WARNING train_rl_overlay_sac_v1 - 2012-04-30 next-month returns are partially missing for optimizer holdings; renormalizing over observed subset for return evaluation. Missing weight: 0.000000
2026-04-29 07:16:27 WARNING train_rl_overlay_sac_v1 - 2012-11-30 next-month returns are partially missing for optimizer holdings; renormalizing over

## Inspect Outputs

In [ ]:
outdir = PROJECT_ROOT / OUTDIR
expected_outputs = [
    outdir / "models/best_model.zip",
    outdir / "models/final_model.zip",
    outdir / "train_history.csv",
    outdir / "validation_metrics.csv",
    outdir / "test_backtest.csv",
    outdir / "test_weights.parquet",
    outdir / "test_action_history.csv",
    outdir / "test_summary.txt",
    outdir / "test_cumret.png",
    outdir / "test_turnover.png",
    outdir / "test_lambda_timeseries.png",
    outdir / "test_tau_timeseries.png",
]

for path in expected_outputs:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:8s} {path}")

summary_path = outdir / "test_summary.txt"
if summary_path.exists():
    print("\n" + summary_path.read_text(encoding="utf-8"))